# Robust Kaggle Smoke Test for OpenVLA (LIBERO-Object)
This notebook verifies that the entire OpenVLA training pipeline (data loading, LoRA fine-tuning, and checkpointing) works correctly in a Kaggle environment.

### 1. Diagnose Kaggle Environment

In [ ]:
import sys
import torch

print("Python Version:", sys.version)
print("PyTorch Version:", torch.__version__)
print("\nCUDA Available:", torch.cuda.is_available())
print("GPU Count:", torch.cuda.device_count())

for i in range(torch.cuda.device_count()):
    print(f"GPU {i}: {torch.cuda.get_device_name(i)} ({torch.cuda.get_device_properties(i).total_memory / 1024**3:.2f} GB)")

if torch.cuda.device_count() == 0:
    print("\n⚠️ WARNING: No GPU detected! This script requires a GPU.")

### 2. Install Exact OpenVLA Dependency Stack
We pin the core dependencies to match OpenVLA's tested stack, while using Kaggle's native TensorFlow (since TF 2.15 is incompatible with Python 3.12). We use `--no-deps` for `tensorflow_graphics` and `dlimp` to prevent them from trying to force older TF versions.

In [ ]:
# Pin core PyTorch/HuggingFace stack
!pip install torch==2.2.0 torchvision==0.17.0 torchaudio==2.2.0 transformers==4.40.1 tokenizers==0.19.1 timm==0.9.10 peft==0.11.1 sentencepiece==0.1.99 draccus==0.8.0

# Install other standard dependencies
!pip install einops jsonlines rich matplotlib huggingface_hub bitsandbytes wandb

# Install TF Data Pipeline
!pip install tensorflow_datasets==4.9.3
!pip install --no-deps tensorflow_graphics==2021.12.3
!pip install --no-deps git+https://github.com/moojink/dlimp_openvla

### 3. Preflight Import Audit
If any of these fail, the environment is broken and training will crash.

In [ ]:
try:
    import torch
    import transformers
    import peft
    import timm
    import tensorflow as tf
    import tensorflow_datasets as tfds
    import tensorflow_graphics
    import dlimp
    import draccus
    import bitsandbytes
    
    print("✅ All critical dependencies imported successfully!")
    print(f"Transformers: {transformers.__version__}")
    print(f"PEFT: {peft.__version__}")
    print(f"TensorFlow: {tf.__version__}")
except ImportError as e:
    print(f"❌ Preflight Import Failed: {e}")
    raise

### 4. Clone OpenVLA & Configure Paths

In [ ]:
!git clone https://github.com/openvla/openvla

import os
# Safely append openvla to PYTHONPATH so its internal imports (like 'prismatic') work
current_pythonpath = os.environ.get('PYTHONPATH', '')
os.environ['PYTHONPATH'] = f"/kaggle/working/openvla:{current_pythonpath}"

### 5. Download the LIBERO-Object RLDS Dataset

In [ ]:
!mkdir -p datasets/openvla/modified_libero_rlds
!huggingface-cli download openvla/modified_libero_rlds \
    --repo-type dataset \
    --include "libero_object_no_noops/*" \
    --local-dir datasets/openvla/modified_libero_rlds

import os
if os.path.exists("datasets/openvla/modified_libero_rlds/libero_object_no_noops"):
    print("✅ Dataset downloaded successfully!")
else:
    print("❌ Dataset directory not found!")

### 6. Run the OpenVLA Smoke Test (Quantized)
We use `--use_quantization True` to ensure the 7B model fits inside the 16GB T4 GPU without instantly causing an Out-Of-Memory (OOM) error.

In [ ]:
%env WANDB_MODE=disabled

!torchrun --standalone --nnodes 1 --nproc-per-node 1 openvla/vla-scripts/finetune.py \
  --vla_path "openvla/openvla-7b" \
  --data_root_dir "datasets/openvla/modified_libero_rlds" \
  --dataset_name "libero_object_no_noops" \
  --run_root_dir "openvla_checkpoints" \
  --adapter_tmp_dir "openvla_checkpoints/tmp" \
  --use_lora True \
  --use_quantization True \
  --lora_rank 32 \
  --batch_size 2 \
  --grad_accumulation_steps 1 \
  --learning_rate 5e-4 \
  --image_aug False \
  --wandb_project "openvla-kaggle-smoke" \
  --wandb_entity "local-test" \
  --save_steps 10 \
  --max_steps 10

### 7. Verify Checkpoint Generation

In [ ]:
import os
import glob

checkpoint_dir = "openvla_checkpoints"
if os.path.exists(checkpoint_dir):
    files = glob.glob(f"{checkpoint_dir}/**/*.pt", recursive=True) + glob.glob(f"{checkpoint_dir}/**/*.safetensors", recursive=True)
    if files:
        print(f"✅ SUCCESS! Found {len(files)} checkpoint files:")
        for f in files:
            size_mb = os.path.getsize(f) / (1024 * 1024)
            print(f" - {f} ({size_mb:.2f} MB)")
    else:
        print("❌ Checkpoint directory exists, but no model weights (.pt or .safetensors) were saved.")
else:
    print("❌ Checkpoint directory was not created. Training likely failed.")